# 07 – Postgres with pg_lake

> Runs against the catalog from `CATALOG` (Lakekeeper, Polaris, RustFS or Unity; default Lakekeeper). What a catalog cannot do is shown by the cell instead of aborting.

Start (the first build takes a while): `docker compose --profile pglake up -d --build pg-lake`

pg_lake makes Iceberg tables usable in Postgres. Reading and writing is done by
`pgduck_server` (embedded DuckDB) with its own static RustFS secret.

- Tables **pg_lake creates itself** are writable and end up in the REST catalog.
- Tables of **other engines** are attached read-only (`read_only = true`).
- Login: on Polaris pg_lake fetches an OAuth token from the catalog; Lakekeeper and Unity run
  without login in the main lab, there pg_lake connects with `rest_auth_type 'none'`.
- Limits per catalog: RustFS does not work (pg_lake cannot do SigV4); on Unity writes are lost
  silently (no `/transactions/commit`); on Lakekeeper 0.15 partitioned tables fail.

In [ ]:
from iceberg_lab.config import PROFILE, REMOTE_SIGNING, SETTINGS, WAREHOUSE_SIGNING, WAREHOUSE_STS, WAREHOUSES
from iceberg_lab.notebook import BUCKET, TRINO_SIGNING, TRINO_STS, catalog_limit, drop_table, header

CATALOG_URI = SETTINGS.catalog_uri
S3_ENDPOINT = SETTINGS.s3_endpoint
header()

In [ ]:
import pandas as pd
import psycopg

from iceberg_lab.engines import PgLakeEngine

pgl = PgLakeEngine(WAREHOUSE_STS)  # creates the server for this catalog (login from the profile)
SERVER = pgl.server
pg = psycopg.connect(host=SETTINGS.pglake_host, port=SETTINGS.pglake_port, user="postgres", autocommit=True)

def sql(q):
    if PgLakeEngine._WRITES.match(q) and not pgl.tx_commit:
        pgl.execute(q)  # an error instead of silent loss: the catalog offers no /transactions/commit
    with pg.cursor() as cur:
        cur.execute(q)
        if cur.description:
            return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])

sql("SELECT extname, extversion FROM pg_extension WHERE extname LIKE 'pg_lake%'")

## Create the catalog server for the catalog

In [ ]:
sql(f"SELECT srvname, srvoptions FROM pg_foreign_server WHERE srvname = '{SERVER}'")

## Create and fill an Iceberg table from Postgres

In [ ]:
sql("CREATE SCHEMA IF NOT EXISTS nb_pglake")
sql("DROP TABLE IF EXISTS nb_pglake.measurements")
sql(f"""
    CREATE TABLE nb_pglake.measurements (station text, at timestamptz, value double precision)
    USING iceberg WITH (catalog = '{SERVER}')
""")
sql(f"""
    INSERT INTO nb_pglake.measurements
    SELECT 'st-' || (g % 3), now() - g * interval '1 hour', random() * 100
    FROM generate_series(1, 500) g
""")
sql("SELECT station, count(*), round(avg(value)::numeric, 2) FROM nb_pglake.measurements GROUP BY 1 ORDER BY 1")

## Mit normalen Postgres-Tabellen joinen

In [ ]:
sql("CREATE TABLE IF NOT EXISTS public.stations (station text PRIMARY KEY, city text)")
sql("INSERT INTO public.stations VALUES ('st-0','Berlin'),('st-1','Cologne'),('st-2','Leipzig') ON CONFLICT DO NOTHING")
sql("""
    SELECT s.city, max(m.value) AS max_value
    FROM nb_pglake.measurements m JOIN public.stations s USING (station)
    GROUP BY 1 ORDER BY 1
""")

## Read the same table from PyIceberg

In [ ]:
from iceberg_lab.engines import pyiceberg_catalog

cat = pyiceberg_catalog(WAREHOUSE_STS)
cat.load_table("nb_pglake.measurements").scan(limit=5).to_pandas()

## Attach another engine's table read-only

In [ ]:
import pyarrow as pa

cat.create_namespace_if_not_exists("nb_pglake_ext")
if cat.table_exists("nb_pglake_ext.prices"):
    drop_table(cat, "nb_pglake_ext.prices")
t = cat.create_table("nb_pglake_ext.prices", schema=pa.schema([("station", pa.string()), ("price", pa.float64())]))
t.append(pa.table({"station": ["st-0", "st-1"], "price": [1.99, 2.49]}))

sql("CREATE SCHEMA IF NOT EXISTS nb_pglake_ext")
sql("DROP TABLE IF EXISTS nb_pglake_ext.prices")
sql(f"""
    CREATE TABLE nb_pglake_ext.prices () USING iceberg
    WITH (catalog = '{SERVER}', read_only = true)
""")
sql("SELECT * FROM nb_pglake_ext.prices")